# WEEK 05 – MID-REVIEW: THE DATA AUDIT
## Moneyball – IPL Player Valuation | Team 05
**Course:** 24ADI204 – Data Science and Visualization  
**Event:** Data-Viz Hack-A-Bit (2026–2027)

### Team members
- Divyesh D S — 25BAD022
- Kavi Prakash M — 25BAD043
- Mithuin P S — 25BAD057
- Madhan P M — 25BAD503
- Renganathan G M — 25BAD087

### Purpose
This notebook prepares the Week 05 mid-review evidence. It summarizes the cleaning decisions completed in Week 03 and the initial statistical findings from Week 04. Week 05 is a review milestone; it does not apply new cleaning, scaling, encoding or feature engineering.

## 1. Review objectives
1. Load the Week 03 cleaned IPL dataset.
2. Verify the dataset shape, missing cells and duplicate rows.
3. Present the documented cleaning decisions and outlier review.
4. Generate descriptive statistics for the cleaned dataset.
5. Reproduce key EDA summaries and visual evidence for the mid-review.
6. Record limitations and next steps.

**Input file expected:** `IPL_Moneyball_Cleaned_Week03.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

sns.set_theme(style="whitegrid")
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Upload IPL_Moneyball_Cleaned_Week03.csv to continue.")

preferred = "IPL_Moneyball_Cleaned_Week03.csv"
file_name = preferred if preferred in csv_files else csv_files[0]
df = pd.read_csv(file_name)

print("Loaded file:", file_name)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Dataset profile and verification
The project dataset has been reported as 3,311 rows and 27 columns, covering 19 IPL seasons from 2008 to 2026 and 794 distinct player values. The Week 03 cleaned version reported 13,099 remaining missing cells and zero duplicate rows.

The next cell verifies these values from the uploaded CSV. If the output differs, investigate the file/version before presenting the review.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Total missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

if "player" in df.columns:
    print("Distinct player values:", df["player"].nunique())
if "season" in df.columns:
    print("Distinct seasons:", df["season"].nunique())
    print("Season range:", df["season"].min(), "to", df["season"].max())

print("\\nData types:")
display(df.dtypes.astype(str).value_counts().rename_axis("Data type").to_frame("Column count"))


## 3. Week 03 cleaning audit – documented decisions

### Missing-value treatment
- Original dataset: 3,311 rows × 27 columns.
- Original missing cells: 17,839.
- Missing `role` values (2,418), `nationality` values (2,296) and `bowl_style` values (26) were replaced with `Unknown`.
- Total cells handled: 4,740.
- Remaining missing cells: 13,099.
- Auction details and numerical performance statistics were retained as missing where their meaning could not safely be inferred. Undefined averages were not filled with zero or a general mean/median.

### Duplicate checking
The original dataset had zero duplicate rows, so no rows were removed.

### Outlier review
Outliers were detected using IQR and Z-score. Detected values were not automatically removed or capped because extreme IPL performance and auction values may be genuine. The counts below are column-level counts, not unique players.

In [ ]:
outlier_audit = pd.DataFrame({
    "Column": ["runs", "strike_rate", "batting_avg", "wickets", "economy", "final_price_lakh"],
    "IQR outliers": [156, 182, 39, 203, 107, 130],
    "Z-score outliers (|Z| > 3)": [46, 20, 25, 44, 31, np.nan]
})
display(outlier_audit)


## 4. Descriptive statistics
The following output is calculated directly from the uploaded cleaned dataset. It should be used as the current mid-review statistical evidence.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
print("Numerical columns:", len(numeric_df.columns))
print("Categorical columns:", len(df.columns) - len(numeric_df.columns))
display(numeric_df.describe().T)


## 5. Missing-value profile
This chart is generated from the uploaded cleaned file. It helps identify columns that still contain missing values. These remaining values were documented rather than filled without a defensible basis.

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_by_column = missing_by_column[missing_by_column > 0]

plt.figure(figsize=(12, 6))
if len(missing_by_column):
    sns.barplot(x=missing_by_column.values, y=missing_by_column.index)
    plt.xlabel("Missing cells")
    plt.ylabel("Column")
    plt.title("Remaining Missing Values by Column – Week 03 Cleaned Dataset")
else:
    plt.text(0.5, 0.5, "No missing values detected", ha="center", va="center")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Initial findings carried forward from Week 04 EDA

Verified Week 04 correlations:
- Runs vs Balls Faced: **0.983936**
- Balls Bowled vs Runs Conceded: **0.982674**
- Balls Bowled vs Matches Bowled: **0.976987**
- Runs vs Fours: **0.964513**

Verified top three players by aggregate runs:
- V Kohli — 9,040
- RG Sharma — 7,183
- S Dhawan — 6,769

These are previously recorded Week 04 results. The charts below display those verified values; they are not recalculated from this file in the absence of a row-level source extract in this preparation.

In [ ]:
corr_data = pd.DataFrame({
    "Variable pair": [
        "Runs vs Balls Faced",
        "Balls Bowled vs Runs Conceded",
        "Balls Bowled vs Matches Bowled",
        "Runs vs Fours"
    ],
    "Correlation": [0.983936, 0.982674, 0.976987, 0.964513]
})

plt.figure(figsize=(9, 4.5))
sns.barplot(data=corr_data, x="Correlation", y="Variable pair")
plt.xlim(0, 1.05)
plt.title("Strongest Verified Positive Correlations – Week 04")
plt.tight_layout()
plt.show()

top3 = pd.DataFrame({
    "Player": ["V Kohli", "RG Sharma", "S Dhawan"],
    "Total runs": [9040, 7183, 6769]
})
plt.figure(figsize=(7, 4))
sns.barplot(data=top3, x="Player", y="Total runs")
plt.title("Top 3 Players by Aggregate Runs – Week 04")
plt.tight_layout()
plt.show()
display(corr_data)
display(top3)


## 7. Review conclusions and limitations

### Conclusions
- The project dataset contains 3,311 player-season records and 27 columns.
- The Week 03 treatment handled 4,740 categorical missing cells using the label `Unknown`.
- The cleaned dataset retained 13,099 missing cells and zero duplicate rows.
- Outliers were identified but retained where there was no evidence of data-entry error.
- Week 04 identified strong positive associations among selected batting and bowling measures.

### Limitations
- Missing auction/profile values may limit analyses that depend on price or player attributes.
- Correlation does not prove causation.
- Aggregate runs alone are not a complete measure of player valuation.
- The original source URLs and full source-data provenance were not established in the materials reviewed.

### Next steps
Week 06 is scheduled for feature engineering: normalization, scaling, categorical encoding and skewness correction. These operations are not performed in this Week 05 review notebook.

## 8. Final audit checklist
Before the mid-review:
- [ ] Upload the correct `IPL_Moneyball_Cleaned_Week03.csv`.
- [ ] Run all cells from top to bottom.
- [ ] Confirm shape is 3,311 × 27.
- [ ] Confirm missing cells are 13,099 and duplicate rows are 0.
- [ ] Save the notebook with outputs.
- [ ] Be prepared to explain why missing values and detected outliers were not indiscriminately removed.
- [ ] Keep the raw and cleaned datasets available for inspection.

**Repository:** https://github.com/Mithuinps/24ADI204-DSV-team-05/tree/main